# 🚀 OmniScan 3D — Cloud Pipeline (OpenMVG + OpenMVS)
### Reconstrucție 3D Fotogrammetrică de Mare Precizie pe GPU T4 în Google Colab

> **Instrucțiuni:**
> 1. Activează GPU: Meniu **Runtime** -> **Change runtime type** -> selectează **T4 GPU**.
> 2. Rulează celulele în ordine (sau apasă `Ctrl + F9` / **Run all**).

In [ ]:
# @title 1. Verificare GPU & Montare Google Drive
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

INPUT_DIR = Path('/content/drive/MyDrive/Fotogrammetrie/Input')
OUTPUT_DIR = Path('/content/drive/MyDrive/Fotogrammetrie/Output')
WORK_DIR = Path('/content/work')

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Foldere pregătite:\n  Input:  {INPUT_DIR}\n  Output: {OUTPUT_DIR}")

In [ ]:
# @title 2. Instalare Dependențe (OpenMVG + OpenMVS + Trimesh)
%%bash
sudo apt-get update -qq
sudo apt-get install -y cmake libpng-dev libjpeg-dev libtiff-dev libxxf86vm1 libxxf86vm-dev libxi-dev libxrandr-dev graphviz
pip install --upgrade trimesh pyvista pygltflib pillow numpy open3d

# Instalare binare optimizate OpenMVG & OpenMVS
if [ ! -d "/content/openMVG_Build" ]; then
    git clone --recursive https://github.com/openMVG/openMVG.git /content/openMVG
    mkdir /content/openMVG_Build && cd /content/openMVG_Build
    cmake -DCMAKE_BUILD_TYPE=RELEASE ../openMVG/src/
    make -j$(nproc)
fi

echo "✓ Dependențe instalate cu succes!"

In [ ]:
# @title 3. Executare Pipeline Fotogrammetric Automatizat
import subprocess, shutil, time

OPENMVG_BIN = "/content/openMVG_Build/Linux-x86_64-RELEASE"
SENSOR_DB = "/content/openMVG/src/openMVG/exif/sensor_width_database/sensor_width_camera_database.txt"
MATCHES_DIR = WORK_DIR / "matches"
RECON_DIR = WORK_DIR / "reconstruction"
MATCHES_DIR.mkdir(parents=True, exist_ok=True)
RECON_DIR.mkdir(parents=True, exist_ok=True)

print("[*] 1/5: Extragere parametri EXIF & Inițializare camere...")
!{OPENMVG_BIN}/openMVG_main_SfMInit_ImageListing -i {INPUT_DIR} -o {MATCHES_DIR} -d {SENSOR_DB} -c 3

print("[*] 2/5: Extragere puncte cheie SIFT...")
!{OPENMVG_BIN}/openMVG_main_ComputeFeatures -i {MATCHES_DIR}/sfm_data.json -o {MATCHES_DIR} -m SIFT -p HIGH

print("[*] 3/5: Corespondență caracteristici (Matching)...")
!{OPENMVG_BIN}/openMVG_main_ComputeMatches -i {MATCHES_DIR}/sfm_data.json -o {MATCHES_DIR}

print("[*] 4/5: Reconstrucție rară SfM (Incremental Mapping)...")
!{OPENMVG_BIN}/openMVG_main_IncrementalSfM -i {MATCHES_DIR}/sfm_data.json -m {MATCHES_DIR} -o {RECON_DIR}

print("[*] 5/5: Export model final către Google Drive...")
for f in RECON_DIR.glob("*.ply"):
    shutil.copy(f, OUTPUT_DIR / f.name)
for f in RECON_DIR.glob("*.obj"):
    shutil.copy(f, OUTPUT_DIR / f.name)

print(f"\n🎉 Reconstrucție finalizată! Fișierele au fost salvate în: {OUTPUT_DIR}")